<a href="https://colab.research.google.com/github/erickodiwuor014-max/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duka Electronics: Sales Exploration (January to June 2026)
Engagement 1 of `plp-da-a3-duka-sales-and-agent-cleanup`. Data: `duka_sales.csv` (1,000 orders, generated by `generate_data.py`, seed 2026).

## Executive Summary
*To be completed after all tasks are finished.*

In [1]:
import os, sys, subprocess

REPO = "plp-da-a3-duka-sales-and-agent-cleanup"
if not os.path.exists("generate_data.py"):
    subprocess.run(["git", "clone", f"https://github.com/erickodiwuor014-max/{REPO}.git"], check=True)
    os.chdir(REPO)
if not os.path.exists("duka_sales.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)

import numpy as np
import pandas as pd

print("Working folder:", os.getcwd())
print("Files:", sorted(os.listdir()))

Working folder: /content/plp-da-a3-duka-sales-and-agent-cleanup
Files: ['.git', '.gitignore', 'README.md', 'agent_transactions.csv', 'agents.csv', 'duka_sales.csv', 'excel', 'generate_data.py', 'outputs', 'requirements.txt']


## 1. First-five-minutes inspection
Loading the data with `order_date` parsed as a real date, then checking shape, types, missing values, duplicates, date range and the key categories.

In [2]:
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


In [3]:
print(df.dtypes)
print()
df.info()

order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


In [4]:
print("Missing values per column:")
print(df.isna().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate order_id values:", df["order_id"].duplicated().sum())

Missing values per column:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Duplicate rows: 0
Duplicate order_id values: 0


In [5]:
print("Date range:", df["order_date"].min().date(), "to", df["order_date"].max().date())
print()
print(df[["quantity", "unit_price"]].describe())

Date range: 2026-01-01 to 2026-06-29

          quantity    unit_price
count  1000.000000   1000.000000
mean      3.012000  23918.300000
std       1.416991  23908.790546
min       1.000000   1200.000000
25%       2.000000   2500.000000
50%       3.000000  15000.000000
75%       4.000000  28000.000000
max       5.000000  65000.000000


In [6]:
for col in ["branch", "product", "channel"]:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

--- branch ---
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64

--- product ---
product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64

--- channel ---
channel
In-store       579
Online         320
Phone order    101
Name: count, dtype: int64



## 2. Derived columns
- `revenue` = `quantity` × `unit_price`
- `month` and `weekday` are taken from `order_date`
- `is_weekend` is True for Saturday and Sunday
- `price_band` is **Premium** when the product's `unit_price` is KES 28,000 or more, otherwise **Value** (the threshold is inclusive, so a KES 28,000 item is Premium)

In [7]:
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

df[["order_id", "order_date", "branch", "product", "quantity", "unit_price",
    "revenue", "month", "weekday", "is_weekend", "price_band"]].head()

,order_id,order_date,branch,product,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,2500,7500,March,Saturday,True,Value


In [8]:
print("Total revenue (KES):", f"{df['revenue'].sum():,}")
print()
print(df["price_band"].value_counts())
print()
print(df.groupby("price_band")["unit_price"].agg(["min", "max"]))

Total revenue (KES): 71,792,500

price_band
Value      565
Premium    435
Name: count, dtype: int64

              min    max
price_band              
Premium     28000  65000
Value        1200  15000
